m# **VRPTW - ALNS + Reinforcement Learning**
**Algorithms:** Standard ALNS , DQN - ALNS

**Dataset:** Solomon Benchmark RC1 + RC2

**Goal:** Show RL-Guided operator selection improves ALNS

# ***1. Setup***

In [5]:
#install, imports
%pip install kagglehub numba safetensors -q

from collections import deque
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from numba import njit

import os, sys, glob, time, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from safetensors.torch import save_file, load_file

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {DEVICE} | Pytorch: {torch.__version__}')

In [ ]:
7

In [6]:
#Download dataset
import kagglehub, shutil

path = kagglehub.dataset_download('senju14/vrptw-benchmark-datasets')
BASE_PATH ='/content/solomon'

if os.path.exists(BASE_PATH):
  shutil.rmtree(BASE_PATH)
shutil.copytree(path, BASE_PATH)

DATA_PATH = os.path.join(BASE_PATH, 'data', 'Solomon')
print(f'Dataset path: {DATA_PATH}')
print(f'Files: {len(glob.glob(os.path.join(DATA_PATH, "*.txt")))}')

: 

# ***2. Config***

In [8]:
@dataclass
class Config:
  #ALNS
  iterations:           int   = 25_000
  destroy_ratio_min:    float = 0.10
  destroy_ratio_max:    float = 0.40
  temp_control:         float = 0.05            #T0 = tmp_control * cost0 / ln2
  temp_min:             float = 0.001
  temp_decay:           float = 0.99975
  sigma1:               int   = 33              # New global best
  sigma2:               int   = 9               # better than current
  sigma3:               int   = 3               # Accepted (worse)
  sigma4:               int   = 1               # Rejected
  weight_decay:         float = 0.10
  segment_size:         int   = 100
  early_stop_patience:  int   = 5_000
  n_runs:               int   = 3

  #RL
  state_dim:            int   = 12
  hidden_dim:           int   = 128
  lr:                   float = 1e-3
  gamma:                float = 0.95
  epsilon_start:        float = 0.40
  epsilon_end:          float = 0.01
  epsilon_decay:        float = 0.9997
  buffer_size:          int   = 8_000
  batch_size:           int   = 64
  target_update_freq:   int   = 200
  train_freq:           int   = 1

  #Reward weights
  reward_vehicle:       float = 5.0
  reward_cost_scale:    float = 1.0
  rweard_best_bonus:    float = 2.0
  reward_infeasible:    float = -3.0

  seed:                 int   = 42

CFG = Config()


# Best Known Solutions (BKS) — Solomon benchmark
BKS = {
    'RC101': {'nv': 14, 'td': 1696.94},
    'RC102': {'nv': 12, 'td': 1554.75},
    'RC103': {'nv': 11, 'td': 1261.67},
    'RC104': {'nv': 10, 'td': 1135.48},
    'RC105': {'nv': 13, 'td': 1629.44},
    'RC106': {'nv': 11, 'td': 1424.73},
    'RC107': {'nv': 11, 'td': 1230.48},
    'RC108': {'nv': 10, 'td': 1139.82},
    'RC201': {'nv': 4 , 'td': 1406.94},
    'RC202': {'nv': 3 , 'td': 1365.64},
    'RC203': {'nv': 3 , 'td': 1049.62},
    'RC204': {'nv': 3 , 'td': 798.46} ,
    'RC205': {'nv': 4 , 'td': 1297.65},
    'RC206': {'nv': 3 , 'td': 1146.32},
    'RC207': {'nv': 3 , 'td': 1061.14},
    'RC208': {'nv': 3 , 'td': 828.14} ,
}

print('Config loaded')
print(f'Iterations: {CFG.iterations:,} | Segment: {CFG.segment_size} | Early stop: {CFG.early_stop_patience:,}')

Config loaded
Iterations: 25,000 | Segment: 100 | Early stop: 5,000


# ***3. Data***

In [ ]:
def load_solomon(path: str) -> Dict:
  with open(path, 'r') as f:
    lines = f.readlines()
  name = lines[0].strip()
  capacity = float(lines[4].strip().split()[1])
